<div style="font-size:30px;font-weight:700;color:#111827;padding-bottom:8px;margin:18px 0;">
보안 로그 자동 분류 경량 LLM
</div>

### 왜 GPT 같은 외부 API가 아니라 "로컬 SLM"인가?

| 이유 | 설명 |
|---|---|
| 🔒 컴플라이언스 | 로그에는 내부 IP, 계정명, 시스템 정보가 포함 → 외부로 전송 불가 |
| 💰 비용 | 하루 수백만 건 × API 요금 = 감당 불가. 로컬 0.5B는 전기료 수준 |
| ⚡ 속도 | 온프레미스에서 실시간 처리 가능 |
| 🎯 작업이 좁다 | "5가지 유형 분류 + 고정 형식 출력"은 소형 모델로 충분 → **파인튜닝이 정답인 상황** |

In [ ]:
%%capture
%pip install -U unsloth

In [ ]:
import unsloth
import torch

In [ ]:
device = torch.accelerator.current_accelerator() if torch.accelerator.is_available() else torch.device("cpu")
device

In [ ]:
total = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU 총 메모리: {total:.1f} GB")

# 4비트 Qwen2.5 모델 로딩

In [ ]:
from unsloth import FastLanguageModel

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-0.5B-Instruct-bnb-4bit",
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)

In [ ]:
# 모델을 불러온 뒤 GPU 메모리를 확인해 봅시다
used = torch.cuda.memory_allocated() / 1024**3
print(f"현재 GPU 메모리 사용량: {used:.2f} GB")

# 학습 데이터 설계 — "논리적으로 의미 있는" 데이터셋의 3가지 조건

실무에서 파인튜닝 데이터셋이 의미를 가지려면:

| 조건 | 이번 데이터셋의 적용 |
|---|---|
| ① 입력→출력 규칙이 일관됨 | 각 공격 유형마다 로그 패턴과 정답 라벨이 1:1로 대응 |
| ② 라벨이 정답임이 보장됨 | 로그를 **규칙으로 생성**하므로 라벨 오류가 0% (실무에선 분석가 검수로 확보) |
| ③ 일반화를 측정할 수 있음 | train 1,000건 / **test 100건 분리**. test는 학습에 안 쓴 새로운 IP·계정·포트 |

### 5가지 이벤트 유형

| event_type | 실제 상황 | severity | 대응(action) |
|---|---|---|---|
| brute_force | SSH 비밀번호 무차별 대입 | high | 출발지 IP 차단, 계정 잠금 확인 |
| port_scan | 방화벽에 다수 포트 접근 탐지 | medium | 방화벽에서 출발지 IP 차단 |
| web_attack | SQL 인젝션 등 웹 공격 시도 | high | WAF 규칙 점검, 해당 요청 차단 |
| c2_beacon | 악성코드의 C2 서버 주기적 통신 | critical | 호스트 격리 및 포렌식 |
| normal | 정상 로그인 | info | 조치 불필요 |

In [ ]:
import random, json

In [ ]:
random.seed(42)   # 재현 가능한 실험을 위해 시드 고정

In [ ]:
USERS  = ["admin", "root", "jhkim", "mspark", "backup", "oracle", "www", "guest"]
HOSTS  = ["web01", "web02", "db01", "app03", "mail01", "vpn01"]
C2DOMS = ["upd-check.xyz", "cdn-sync.top", "stat-push.club", "img-cache.icu"]

In [ ]:
def rand_ip():
    return f"{random.randint(11,223)}.{random.randint(0,255)}.{random.randint(0,255)}.{random.randint(1,254)}"

In [ ]:
def gen_brute_force():
    ip, user, host = rand_ip(), random.choice(USERS), random.choice(HOSTS)
    lines = [
        f"{host} sshd[{random.randint(1000,9999)}]: Failed password for {user} "
        f"from {ip} port {random.randint(30000,65000)} ssh2"
        for _ in range(3)
    ]
    return "\n".join(lines)

In [ ]:
def gen_port_scan():
    ip, host = rand_ip(), random.choice(HOSTS)
    ports = random.sample([21, 22, 23, 25, 80, 110, 135, 139, 443, 445, 1433, 3306, 3389, 8080], 4)
    lines = [
        f"{host} firewall: DENY TCP {ip}:{random.randint(40000,65000)} -> 10.0.0.{random.randint(2,254)}:{p}"
        for p in ports
    ]
    return "\n".join(lines)

In [ ]:
def gen_web_attack():
    ip, host = rand_ip(), random.choice(HOSTS)
    payload = random.choice([
        "id=1' OR '1'='1", "q=admin'--", "u=1 UNION SELECT password FROM users",
        "file=../../etc/passwd", "name=<script>alert(1)</script>",
    ])
    return (f'{host} nginx: {ip} - - "GET /login.php?{payload} HTTP/1.1" '
            f'{random.choice([200, 403, 500])} {random.randint(200,5000)}')

In [ ]:
def gen_c2_beacon():
    host, dom = random.choice(HOSTS), random.choice(C2DOMS)
    lines = [
        f"{host} dns: query {dom} from 10.0.0.{random.randint(2,254)} (interval 60s)"
        for _ in range(3)
    ]
    return "\n".join(lines)

In [ ]:
def gen_normal():
    ip, user, host = rand_ip(), random.choice(USERS), random.choice(HOSTS)
    return (f"{host} sshd[{random.randint(1000,9999)}]: Accepted password for {user} "
            f"from {ip} port {random.randint(30000,65000)} ssh2")

In [ ]:
# 유형별 (로그 생성 함수, severity, action) 정의 — 이 표가 곧 "정답 규칙"
RULES = {
    "brute_force": (gen_brute_force, "high",     "출발지 IP 차단 및 계정 잠금 확인"),
    "port_scan":   (gen_port_scan,   "medium",   "방화벽에서 출발지 IP 차단"),
    "web_attack":  (gen_web_attack,  "high",     "WAF 규칙 점검 및 해당 요청 차단"),
    "c2_beacon":   (gen_c2_beacon,   "critical", "감염 의심 호스트 격리 및 포렌식"),
    "normal":      (gen_normal,      "info",     "조치 불필요"),
}

In [ ]:
def make_sample():
    etype = random.choice(list(RULES.keys()))
    gen_fn, severity, action = RULES[etype]
    label = {"event_type": etype, "severity": severity, "action": action}
    return {"log": gen_fn(), "label": label}

In [ ]:
# train 1,000건 / test 100건 — test는 학습에 절대 사용하지 않음!
train_data = [make_sample() for _ in range(1000)]
test_data  = [make_sample() for _ in range(100)]
print("train:", len(train_data), "| test:", len(test_data))

In [ ]:
idx_test = 0
train_data[idx_test]

In [ ]:
print("[로그]\n" + train_data[idx_test]["log"])
print("\n[정답 라벨]\n" + json.dumps(train_data[idx_test]["label"], ensure_ascii=False, indent=2))

# 프롬프트와 평가 함수 정의

### 공정한 비교의 핵심: 파인튜닝 전에도 "똑같은 지시문"을 줍니다
허용 라벨 목록까지 알려주는 친절한 지시문을 학습 전/후 동일하게 사용합니다.
그래야 "지시문이 부족해서 못한 것"이 아니라 **"지시만으로는 안 되던 것을
파인튜닝이 해결했다"** 는 논리가 성립합니다.

### 평가 지표 3가지 (test 100건 중 50건 사용)
1. **JSON 준수율**: 답변에서 유효한 JSON을 꺼낼 수 있는가 (자동화 파이프라인의 전제조건)
2. **event_type 정확도**: 공격 유형을 맞혔는가
3. **severity 정확도**: 심각도를 맞혔는가


In [ ]:
import re

In [ ]:
def build_prompt(log):
    return (
        "당신은 보안관제(SOC) 분석 도우미입니다. 아래 보안 로그를 분석해 JSON 한 개로만 답하세요.\n"
        '형식: {"event_type": "...", "severity": "...", "action": "..."}\n'
        "event_type은 [brute_force, port_scan, web_attack, c2_beacon, normal] 중 하나,\n"
        "severity는 [critical, high, medium, info] 중 하나입니다.\n\n"
        "[로그]\n" + log
    )

In [ ]:
def analyze(log, max_length=1024):
    """로그 → 모델 답변 (그리디 디코딩: 항상 같은 답 → 공정한 비교)"""
    messages = [{"role": "user", "content": build_prompt(log)}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True,
        return_tensors="pt", return_dict=True,
    ).to("cuda")
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length = max_length,
            do_sample      = False,
            pad_token_id   = tokenizer.pad_token_id or tokenizer.eos_token_id,
        )
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

In [ ]:
def extract_json(text):
    """답변에서 첫 번째 JSON 객체를 추출. 실패하면 None"""
    match = re.search(r"\{.*?\}", text, re.DOTALL)
    if not match:
        return None
    try:
        return json.loads(match.group())
    except json.JSONDecodeError:
        return None

In [ ]:
def evaluate(samples, name=""):
    """test 샘플들에 대해 3가지 지표 + 유형별 정확도 계산"""
    json_ok = type_ok = sev_ok = 0
    per_type = {t: [0, 0] for t in RULES}   # {유형: [맞힘, 전체]}
    for i, s in enumerate(samples):
        answer = analyze(s["log"])
        parsed = extract_json(answer)
        gt = s["label"]
        per_type[gt["event_type"]][1] += 1
        if parsed is not None:
            json_ok += 1
            if parsed.get("event_type") == gt["event_type"]:
                type_ok += 1
                per_type[gt["event_type"]][0] += 1
            if parsed.get("severity") == gt["severity"]:
                sev_ok += 1
        print(f"\r{name} 평가 중... {i+1}/{len(samples)}", end="")
    print()
    n = len(samples)
    return {
        "JSON 준수율":       100.0 * json_ok / n,
        "event_type 정확도": 100.0 * type_ok / n,
        "severity 정확도":   100.0 * sev_ok / n,
        "유형별": {t: (100.0 * c / tot if tot else 0.0) for t, (c, tot) in per_type.items()},
    }

# 파인튜닝 전(baseline) 성능 측정

먼저 원본 모델의 실력을 재둡니다. 시간 절약을 위해 test 50건만 사용합니다 (약 2~3분).

0.5B 모델은 지시문을 줘도 형식을 어기거나 라벨을 틀리는 경우가 많을 것입니다.

In [ ]:
FastLanguageModel.for_inference(model)

In [ ]:
eval_set = test_data[:50]

In [ ]:
# 먼저 답변 예시 2개를 눈으로 확인
for s in eval_set[:2]:
    print("[로그]", s["log"].splitlines()[0][:80], "...")
    print("[정답]", s["label"]["event_type"], "/", s["label"]["severity"])
    print("[학습 전 답변]", analyze(s["log"])[:200])
    print("-" * 70)

In [ ]:
before_metrics = evaluate(eval_set, name="학습 전")

for k in ["JSON 준수율", "event_type 정확도", "severity 정확도"]:
    print(f"{k:<20}: {before_metrics[k]:5.1f}%")

## 5단계. LoRA 부착 + 학습 데이터 변환

train 1,000건을 채팅 형식으로 변환합니다. 정답 JSON이 assistant의 답변이 됩니다.


In [ ]:
from datasets import Dataset

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha    = 16,
    lora_dropout  = 0,
    bias          = "none",
    use_gradient_checkpointing = "unsloth",
    random_state  = 42,
)
model.print_trainable_parameters()

In [ ]:
def to_chat_text(sample):
    messages = [
        {"role": "user",      "content": build_prompt(sample["log"])},
        {"role": "assistant", "content": json.dumps(sample["label"], ensure_ascii=False)},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)

train_dataset = Dataset.from_dict({"text": [to_chat_text(s) for s in train_data]})

In [ ]:
print(train_dataset[0]["text"])

## 6단계. 파인튜닝 실행

v2와 동일한 설정에, 답변 토큰에만 loss를 계산하는 `train_on_responses_only`를 적용합니다.
80스텝 × 배치 8 = 640건 학습, T4에서 약 4~6분.





## 7단계. 파인튜닝 실행

### 💡 v1과 달라진 중요한 설정: 답변 부분만 학습 (`train_on_responses_only`)

v1에서는 사용자 **질문까지 포함해서** loss를 계산했습니다.
그러면 모델이 "질문을 생성하는 법"까지 배우느라 학습이 흐려집니다.

이번에는 **assistant 답변 토큰에만 loss를 계산**합니다.
→ "이 질문이 오면 이렇게 답해"라는 목표에 학습이 집중됩니다.

| 설정 | 값 | 의미 |
|---|---|---|
| max_steps | 60 | 배치 8 × 60 = 샘플 480개 학습 (약 3~5분) |
| learning_rate | 2e-4 | 말투 같은 표면 패턴 학습에 적당한 값 |


In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

In [ ]:
trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = train_dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        max_seq_length     = 1024,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        max_steps          = 80,
        warmup_steps       = 5,
        learning_rate      = 2e-4,
        logging_steps      = 10,
        optim              = "adamw_8bit",
        weight_decay       = 0.01,
        lr_scheduler_type  = "linear",
        seed               = 42,
        output_dir         = "outputs",
        report_to          = "none",
    ),
)

In [ ]:
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>system\n",
    response_part    = "<|im_start|>assistant\n",
)

In [ ]:
trainer_stats = trainer.train()

## 7단계. 파인튜닝 후 성능 측정 — 같은 test 50건, 같은 조건

**학습에 한 번도 쓰이지 않은** 로그로 평가하므로,
점수가 오른다면 모델이 패턴을 **일반화**했다는 뜻입니다 (외운 것이 아니라!).


In [ ]:
FastLanguageModel.for_inference(model)

In [ ]:
# 같은 예시 2개를 먼저 눈으로 확인
for s in eval_set[:2]:
    print("[로그]", s["log"].splitlines()[0][:80], "...")
    print("[정답]", s["label"]["event_type"], "/", s["label"]["severity"])
    print("[학습 후 답변]", analyze(s["log"])[:200])
    print("-" * 70)

In [ ]:
after_metrics = evaluate(eval_set, name="학습 후")

print(f"\n{'지표':<20} {'학습 전':>8} {'학습 후':>8}")
print("-" * 40)
for k in ["JSON 준수율", "event_type 정확도", "severity 정확도"]:
    print(f"{k:<20} {before_metrics[k]:>7.1f}% {after_metrics[k]:>7.1f}%")

print(f"\n{'유형별 event_type 정확도':<24} {'학습 전':>8} {'학습 후':>8}")
print("-" * 44)
for t in RULES:
    print(f"{t:<24} {before_metrics['유형별'][t]:>7.1f}% {after_metrics['유형별'][t]:>7.1f}%")

In [ ]:
# (선택) 모델 저장
model.save_pretrained("soc_triage_lora")
tokenizer.save_pretrained("soc_triage_lora")
!ls -lh soc_triage_lora